In [ ]:
import os
import re
import time
from urllib.parse import urlparse

import pandas as pd
import requests_cache
from dotenv import load_dotenv
from lxml import etree

load_dotenv()

UA = os.getenv('SCRAPER_CONTACT_UA')

session = requests_cache.CachedSession(
    'd20pfsrd_cache',
    backend='sqlite',
    expire_after=-1,
    allowable_codes=(200, 404),  # d20pfsrd sometimes 404s with valid sitemap bodies
)

HEADERS = {'User-Agent': UA}
DELAY = 1.0


def get(url):
    resp = session.get(url, headers=HEADERS)
    if not getattr(resp, 'from_cache', False):
        time.sleep(DELAY)
    looks_like_xml = b'<urlset' in resp.content[:500] or b'<sitemapindex' in resp.content[:500]
    if resp.status_code != 200 and not looks_like_xml:
        resp.raise_for_status()
    return resp


def get_locs(url):
    resp = get(url)
    root = etree.fromstring(resp.content)
    return root.xpath('//*[local-name()="loc"]/text()')

In [ ]:
index = get_locs('https://www.d20pfsrd.com/wp-sitemap.xml')
page_sitemaps = [u for u in index if 'posts-page' in u]

In [ ]:
page_indices = {}
for url in page_sitemaps:
    n = int(re.search(r'page-(\d+)\.xml', url).group(1))
    page_indices[n] = get_locs(url)
    print(n, len(page_indices[n]), page_indices[n][:3])

In [ ]:
rows = []
for i, url in page_indices.items():
    for u in url:
        rows.append({'sitemap_page': i, 'url': u})

df = pd.DataFrame(rows)

path = df['url'].apply(lambda u: urlparse(u).path.strip('/'))
df['depth'] = path.apply(lambda p: len(p.split('/')) if p else 0)
segments = path.apply(lambda p: p.split('/') if p else [])
df['level_1'] = segments.apply(lambda s: s[0] if len(s) > 0 else '')
df['level_2'] = segments.apply(lambda s: s[1] if len(s) > 1 else '')
df['level_3'] = segments.apply(lambda s: s[2] if len(s) > 2 else '')
df['level_4'] = segments.apply(lambda s: s[3] if len(s) > 3 else '')
df['level_5'] = segments.apply(lambda s: s[4] if len(s) > 4 else '')
df['level_6'] = segments.apply(lambda s: s[5] if len(s) > 5 else '')
df['level_7'] = segments.apply(lambda s: s[6] if len(s) > 6 else '')
df['level_8'] = segments.apply(lambda s: s[7] if len(s) > 7 else '')

In [ ]:
counts = df.groupby(['level_1', 'level_2']).size().reset_index(name='count')
counts = counts.sort_values('count', ascending=False)
counts

In [ ]:
# 'work-area',                  # private staging
# 'subscribe',                  # Open Gaming Network Subscription advert
# 'extras',                     # mostly homebrew, scope only official
# 'alternative-rule-systems',   # scope only base rules

df_alignment_description = df[df['level_1'] == 'alignment-description'].copy()
df_basics_ability_scores = df[df['level_1'] == 'basics-ability-scores'].copy()
df_bestiary = df[df['level_1'] == 'bestiary'].copy()
df_classes = df[df['level_1'] == 'classes'].copy()
df_equipment = df[df['level_1'] == 'equipment'].copy()
df_feats = df[df['level_1'] == 'feats'].copy()
df_gamemastering = df[df['level_1'] == 'gamemastering'].copy()
df_magic = df[df['level_1'] == 'magic'].copy()
df_magic_items = df[df['level_1'] == 'magic-items'].copy()
df_races = df[df['level_1'] == 'races'].copy()
df_skills = df[df['level_1'] == 'skills'].copy()
df_traits = df[df['level_1'] == 'traits'].copy()

In [ ]:
third_party_publishers = [
    '3rd-party-publisher',
    '3rd-party-publishers',
    '4-winds-fantasy-gaming',
    'adamant-entertainment',
    'ascension-games',
    'ascension-games-llc',
    'azoth-games',
    'bloodstone-press',
    'd20pfsrd-com-publishing',
    'dreamscarred-press',
    'drop-dead-studios',
    'everyman-gaming',
    'everyman-gaming-llc',
    'far-distant-future-publishing',
    'fat-goblin-games',
    'fire-mountain-games',
    'flaming-crab-games',
    'flying-pincushion-games',
    'forest-guardian-press',
    'frog-god-games',
    'icosa-entertainment-llc',
    'jon-brazer-enterprises',
    'jon-brazer-enterprizes',
    'kobold-press',
    'kobold-press-open-design-llc',
    'legendary-games',
    'little-red-goblin-games',
    'little-red-goblin-games-llc',
    'louis-porter-jr-design',
    'michael-mars',
    'misfit-studios',
    'names-games',
    'necromancers-of-the-northwest',
    'onyx-path-and-nocturnal-publishing',
    'orphaned-bookworm-productions',
    'paizo-fans-united',
    'petersen-games',
    'purple-duck-games',
    'radiance-house',
    'rite-publishing',
    'rogue-genius-games',
    'samurai-sheepdog',
    'shm-publishing',
    'spes-magna-games',
    'studio-m',
    'the-knotty-works',
    'total-party-kill-games',
    'varyags-forge',
    'xoth-net-publishing',
    'james-ray',
    'librarians-leviathans',
    'arcanist-exploits-3rd-party',
]

third_party_terms = [
    '3rd-party',
    '3pp',
    '4-winds-fantasy-gaming',
]

In [ ]:
# filter df_alignment_description
df_alignment_description = df_alignment_description[~df_alignment_description['level_2'].isin([''])]  # top level list

In [ ]:
# filter df_basics_ability_scores
df_basics_ability_scores = df_basics_ability_scores[
    ~df_basics_ability_scores['url'].isin(
        [
            'https://www.d20pfsrd.com/basics-ability-scores/',  # top level list
            'https://www.d20pfsrd.com/basics-ability-scores/more-character-options/',  # top level list
        ]
    )
]

In [ ]:
# filter df_bestiary
bestiary_level2_excludes = [
    'bestiary-alphabetical',  # lists
    'bestiary-by-challenge-rating',  # lists
    'bestiary-by-terrain',  # lists
    'bestiary-hub',  # lists
    'fan-conversions',  # scope only official
    'indexes-and-tables',  # index monsters by CR/locations
    'tools',  # interactive tools
]

df_bestiary = df_bestiary[~df_bestiary['level_2'].isin(bestiary_level2_excludes)].copy()


# individual URL excludes
bestiary_url_excludes = [
    'https://www.d20pfsrd.com/bestiary/',  # top level
    'https://www.d20pfsrd.com/bestiary/rules-for-monsters/',  # list
    'https://www.d20pfsrd.com/bestiary/rules-for-monsters/monster-roles/',  # list
    'https://www.d20pfsrd.com/bestiary/rules-for-monsters/creature-types/new-creature-subtypes/',  # third party
    'https://www.d20pfsrd.com/bestiary/rules-for-monsters/universal-monster-rules/umr-3pp-frog-god-games/',  # third party
    'https://www.d20pfsrd.com/bestiary/unique-monsters/',  # list
    'https://www.d20pfsrd.com/bestiary/unique-monsters/under-cr-1/',  # list
    'https://www.d20pfsrd.com/bestiary/npc-s/npc-db/',  # excel table
    'https://www.d20pfsrd.com/bestiary/monster-listings/',  # list of monsters by type
    'https://www.d20pfsrd.com/classes/',  # top level
    'https://www.d20pfsrd.com/classes/core-classes/',  # list
]

# exclude bestiary lists by cr
for cr in range(1, 22):
    bestiary_url_excludes.append(f'https://www.d20pfsrd.com/bestiary/npc-s/npcs-cr-{cr}/')
for cr in range(1, 26):
    bestiary_url_excludes.append(f'https://www.d20pfsrd.com/bestiary/unique-monsters/cr-{cr}/')


df_bestiary = df_bestiary[(~df_bestiary['url'].isin(bestiary_url_excludes))]


pattern = '|'.join(re.escape(p) for p in third_party_publishers + third_party_terms)
mask = df_bestiary['url'].str.contains(pattern, na=False, regex=True)
df_bestiary = df_bestiary[~mask].copy()

In [ ]:
# filter df_classes
classes_level2_include = [
    'core-classes',
    '',
    # '3rd-party-classes', #scope only official
    # '3rd-party-npc-classes', #scope only official
    # '3rd-party-prestige-classes', #scope only official
    'alternate-classes',
    'base-classes',
    'character-advancement',
    'class-archetypes',
    # 'arcane-archetypes-rogue-genius-games', #scope only official
    'hybrid-classes',
    'monster-classes',
    'npc-classes',
    'prestige-classes',
    'unchained-classes',
    'ex-class-archetypes',
]


classes_level3_exclude = [
    '3rd-party-alternate-classes',  # scope only official
    '3rd-party-hybrid-classes',  # scope only official
    'hill-giant-monster-class',  # scope only official
    'lizardfolk-monster-class',  # scope only official
    'astral-deva',  # scope only official
]

classes_level4_exclude = [
    '3rd-party-shaman-spirits',
    'arcanist-greater-exploits',
    'archetypes-bloodstone-press',
    'archetypes-orphaned-bookworm-productions-2',
    'fighter-bravery-alternative-options',
    'gods-3rd-party-publishers',
    'hexes-3rd-party-publishers',
    'unchained-barbarian-archetypes-shm-publishing',
    'vampire-hunter-archetypes-legendary-games',
    'witch-patrons-3rd-party-publishers',
]

classes_level6_exclude = [
    'animal-companion-archetypes-by-other-publishers',
]


df_classes = df_classes[
    (df_classes['level_2'].isin(classes_level2_include))
    & (~df_classes['level_3'].isin(classes_level3_exclude))
    & (~df_classes['level_4'].isin(classes_level4_exclude))
    & (~df_classes['level_6'].isin(classes_level6_exclude))
]


pattern = '|'.join(re.escape(p) for p in third_party_publishers + third_party_terms)
df_classes = df_classes[~df_classes['level_5'].str.contains(pattern, na=False, regex=True)].copy()

In [ ]:
# filter df_equipment
equipment_level2_exclude = ['3rd-party-equipment', 'special-materials-third-party']
df_equipment = df_equipment[~df_equipment['level_2'].isin(equipment_level2_exclude)]

pattern = '|'.join(re.escape(p) for p in third_party_publishers + third_party_terms)
mask = df_equipment['url'].str.contains(pattern, na=False, regex=True)
df_equipment = df_equipment[~mask].copy()

In [ ]:
# filter df_feats
pattern = '|'.join(re.escape(p) for p in third_party_publishers + third_party_terms)
mask = df_feats['url'].str.contains(pattern, na=False, regex=True)
df_feats = df_feats[~mask].copy()

In [ ]:
# filter df_gamemastering
gamemastering_level2_exclude = [
    'tools',
]

df_gamemastering = df_gamemastering[~df_gamemastering['level_2'].isin(gamemastering_level2_exclude)]


pattern = '|'.join(re.escape(p) for p in third_party_publishers + third_party_terms)
mask = df_gamemastering['level_3'].str.contains(pattern, na=False, regex=True) | df_gamemastering[
    'level_4'
].str.contains(pattern, na=False, regex=True)

df_gamemastering = df_gamemastering[~mask]

In [ ]:
# filter df_magic
pattern = '|'.join(re.escape(p) for p in third_party_publishers + third_party_terms)
mask = df_magic['url'].str.contains(pattern, na=False, regex=True)
df_magic = df_magic[~mask].copy()

df_magic = df_magic[~df_magic['level_2'].isin(['tools'])]

# lists, tools
magic_urls_excludes = [
    'https://www.d20pfsrd.com/magic/all-spells/',
    'https://www.d20pfsrd.com/magic/spell-lists-and-domains/',
    'https://www.d20pfsrd.com/magic/tools/',
    'https://www.d20pfsrd.com/magic/variant-magic-rules/',
    'https://www.d20pfsrd.com/magic/all-spells/a/',
    'https://www.d20pfsrd.com/magic/all-spells/b/',
    'https://www.d20pfsrd.com/magic/all-spells/c/',
    'https://www.d20pfsrd.com/magic/all-spells/d/',
    'https://www.d20pfsrd.com/magic/all-spells/e/',
    'https://www.d20pfsrd.com/magic/all-spells/f/',
    'https://www.d20pfsrd.com/magic/all-spells/g/',
    'https://www.d20pfsrd.com/magic/all-spells/h/',
    'https://www.d20pfsrd.com/magic/all-spells/i/',
    'https://www.d20pfsrd.com/magic/all-spells/j/',
    'https://www.d20pfsrd.com/magic/all-spells/k/',
    'https://www.d20pfsrd.com/magic/all-spells/l/',
    'https://www.d20pfsrd.com/magic/all-spells/m/',
    'https://www.d20pfsrd.com/magic/all-spells/n/',
    'https://www.d20pfsrd.com/magic/all-spells/o/',
    'https://www.d20pfsrd.com/magic/all-spells/p/',
    'https://www.d20pfsrd.com/magic/all-spells/q/',
    'https://www.d20pfsrd.com/magic/all-spells/r/',
    'https://www.d20pfsrd.com/magic/all-spells/s/',
    'https://www.d20pfsrd.com/magic/all-spells/spells-a-d/',
    'https://www.d20pfsrd.com/magic/all-spells/spells-e-h/',
    'https://www.d20pfsrd.com/magic/all-spells/spells-i-l/',
    'https://www.d20pfsrd.com/magic/all-spells/spells-m-p/',
    'https://www.d20pfsrd.com/magic/all-spells/spells-q-t/',
    'https://www.d20pfsrd.com/magic/all-spells/spells-u-z/',
    'https://www.d20pfsrd.com/magic/all-spells/t/',
    'https://www.d20pfsrd.com/magic/all-spells/u/',
    'https://www.d20pfsrd.com/magic/all-spells/v/',
    'https://www.d20pfsrd.com/magic/all-spells/w/',
    'https://www.d20pfsrd.com/magic/all-spells/x/',
    'https://www.d20pfsrd.com/magic/all-spells/y/',
    'https://www.d20pfsrd.com/magic/all-spells/z/',
]

df_magic = df_magic[(~df_magic['url'].isin(magic_urls_excludes))]

In [ ]:
# df_magic_items
pattern = '|'.join(re.escape(p) for p in third_party_publishers + third_party_terms)
mask = df_magic_items['url'].str.contains(pattern, na=False, regex=True)
df_magic_items = df_magic_items[~mask].copy()


magic_items_url_excludes = ['https://www.d20pfsrd.com/magic-items/magic-items-db/']
df_magic_items = df_magic_items[(~df_magic_items['url'].isin(magic_items_url_excludes))]

In [ ]:
# df_races
pattern = '|'.join(re.escape(p) for p in third_party_publishers + third_party_terms)
mask = df_races['url'].str.contains(pattern, na=False, regex=True)
df_races = df_races[~mask].copy()

In [ ]:
# df_skills
pattern = '|'.join(re.escape(p) for p in third_party_publishers + third_party_terms)
mask = df_skills['url'].str.contains(pattern, na=False, regex=True)
df_skills = df_skills[~mask].copy()
skills_urls_excludes = ['https://www.d20pfsrd.com/skills/skills-from-other-publishers/']
df_skills = df_skills[(~df_skills['url'].isin(skills_urls_excludes))]

In [ ]:
# df_traits
pattern = '|'.join(re.escape(p) for p in third_party_publishers + third_party_terms)
mask = df_traits['url'].str.contains(pattern, na=False, regex=True)
df_traits = df_traits[~mask].copy()

traits_level2_exclude = [
    'campaign-traits',
    'tools',
]

df_traits = df_traits[~df_traits['level_2'].isin(traits_level2_exclude)]

In [ ]:
df_final = pd.concat(
    [
        df_alignment_description,
        df_basics_ability_scores,
        df_bestiary,
        df_classes,
        df_equipment,
        df_feats,
        df_gamemastering,
        df_magic,
        df_magic_items,
        df_races,
        df_skills,
        df_traits,
    ],
    ignore_index=True,
)

print(len(df_final))

In [ ]:
print(len(df))

In [ ]:
len(df) - len(df_final)

In [ ]:
counts = df_final.groupby(['level_1', 'level_2']).size().reset_index(name='count')
counts = counts.sort_values('count', ascending=False)
counts

In [ ]:
df_final.level_1.value_counts()

In [ ]:
df_classes['level_2'].unique()

In [ ]:
df_final.to_parquet('d20pfsrd_links.parquet')

In [ ]:
df_final.head(20).to_parquet('d20pfsrd_links_20test.parquet')